# Information Retrieval with BM25 and Embeddings
by Mickey Choy and Yuheng Ouyang

## 0. Imports and helper functions

In [1]:
import gzip
import json
import pickle
import re
import string
import pandas as pd
from IPython.display import display, Markdown
from textwrap import shorten

# Get the root
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

#import nltk
#nltk.download("stopwords")
from nltk.corpus import stopwords
from rank_bm25 import BM25Okapi
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer

# Import scripts 
from src.bm25 import build_bm25, save_bm25, load_bm25, bm25_search
from src.semantic import (
    build_embeddings,
    build_faiss_index,
    save_faiss,
    load_faiss,
    semantic_search
)

In [2]:
def load_jsonl_gz(path, n=5000):
    '''
    load first N records from a .jsonl.gz file
    '''
    records = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return records

In [3]:
def display_search_results(results, query, system_name, max_chars=200):
    '''
    Pretty display for retrieval results.
    '''
    rows = []

    for rank, r in enumerate(results, start=1):
        metadata = r["metadata"]
        source = metadata["source"]

        raw_text = r["text"].strip()
        parts = [p.strip() for p in raw_text.split("\n\n") if p.strip()]

        title = parts[0]
        body = " ".join(parts[1:]) if len(parts) > 1 else raw_text
        body = " ".join(body.split())

        if source == "metadata":
            categories = metadata.get("categories", [])
            categories_str = " > ".join(categories[:4]) if isinstance(
                categories, list) else ""
            info = metadata.get("main_category", "")
            if categories_str:
                info = f"{info} | {categories_str}" if info else categories_str
        else:
            info = f'rating={metadata.get("rating", "")}, verified={metadata.get("verified_purchase", "")}'

        rows.append({
            "Rank": rank,
            "Score": float(r["score"]),
            "Source": source,
            "ASIN": metadata["asin"],
            "Title": title,
            "Info": info,
            "Snippet": shorten(body, width=max_chars, placeholder="...")
        })

    df = pd.DataFrame(rows)

    title = f"#### {system_name} results for: '{query}'"
    display(Markdown(title))

    display(
        df.style
        .hide(axis="index")
        .format({"Score": "{:.3f}"})
        .set_properties(subset=["Title", "Info", "Snippet"], **{"text-align": "left"})
        .set_properties(subset=["Snippet"], **{"white-space": "pre-wrap"})
    )

    return df

## 1. Data exploration and preprocessing

In [4]:
# Load small subsets (5000 rows)
meta_path = "../data/raw/meta_Appliances.jsonl.gz"
review_path = "../data/raw/Appliances.jsonl.gz"

meta_records = load_jsonl_gz(meta_path, n=5000)
review_records = load_jsonl_gz(review_path, n=5000)

print(f"Loaded {len(meta_records)} metadata records")
print(f"Loaded {len(review_records)} review records")

# Convert to DataFrames for easier EDA
meta_df = pd.DataFrame(meta_records)
review_df = pd.DataFrame(review_records)

meta_df.head()

Loaded 5000 metadata records
Loaded 5000 review records


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Industrial & Scientific,"ROVSUN Ice Maker Machine Countertop, Make 44lb...",3.7,61,[【Quick Ice Making】This countertop ice machine...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Our Point of View on the Euhomy Ic...,ROVSUN,"[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': 'ROVSUN', 'Model Name': 'ICM-2005', ...",B08Z743RRD,None
1,Tools & Home Improvement,"HANSGO Egg Holder for Refrigerator, Deviled Eg...",4.2,75,"[Plastic, Practical Kitchen Storage - Our egg ...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '10 Eggs Egg Holder for Refrigerato...,HANSGO,"[Appliances, Parts & Accessories, Refrigerator...","{'Manufacturer': 'HANSGO', 'Part Number': 'HAN...",B097BQDGHJ,None
2,Tools & Home Improvement,"Clothes Dryer Drum Slide, General Electric, Ho...",3.5,18,[],"[Brand new dryer drum slide, replaces General ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],GE,"[Appliances, Parts & Accessories]","{'Manufacturer': 'RPI', 'Part Number': 'WE1M33...",B00IN9AGAE,None
3,Tools & Home Improvement,154567702 Dishwasher Lower Wash Arm Assembly f...,4.5,26,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],folosem,"[Appliances, Parts & Accessories, Dryer Parts ...","{'Manufacturer': 'folosem', 'Part Number': '15...",B0C7K98JZS,None
4,Tools & Home Improvement,Whirlpool W10918546 Igniter,3.8,12,[This is a Genuine OEM Replacement Part.],[Whirlpool Igniter],25.07,[{'thumb': 'https://m.media-amazon.com/images/...,[],Whirlpool,"[Appliances, Parts & Accessories]","{'Manufacturer': 'Whirlpool', 'Part Number': '...",B07QZHQTVJ,None


### 1.1. Dataset overview

In [5]:
print("Metadata fields:", meta_df.columns.tolist())
print("Review fields:", review_df.columns.tolist())
print()
print("Metadata shape:", meta_df.shape)
print("Review shape:", review_df.shape)

Metadata fields: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
Review fields: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Metadata shape: (5000, 14)
Review shape: (5000, 10)


### 1.2. Sample records

In [6]:
meta_df.sample(3)
review_df.sample(3)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
2762,5.0,Nice Finish For Range Top,The Corelle Coordinates Burner Covers match my...,[],B000UOG7CG,B000UOG7CG,AH5UPJBD6NRFYMYP2JZF45UZ5PFA,1276389587000,0,True
4303,4.0,Delivered as ordered.,Works as expected.,[],B07S7WF3S3,B0C64H22TB,AFNG4ICJNWPS2LTHHFZHYCCSVOPQ,1630090195377,0,True
2835,5.0,Convenient,Convenient,[],B0728JN794,B09W8W65XY,AHDLTVY2XJMUSPXQAPRNFZRLODRA,1558548087461,0,True


### 1.3. Data preprocessing

In [7]:
# Fields we decided to use for retrieval
META_TEXT_FIELDS = ["title", "description", "features"]
REVIEW_TEXT_FIELDS = ["title", "text"]

documents = []

# Process metadata entries
for _, row in meta_df.iterrows():
    text_parts = []

    for field in META_TEXT_FIELDS:
        value = row.get(field)
        if isinstance(value, str) and len(value.strip()) > 0:
            text_parts.append(value.strip())

    # Skip if no meaningful text
    if len(text_parts) == 0:
        continue

    full_text = "\n\n".join(text_parts)

    documents.append(
        Document(
            page_content=full_text,
            metadata={
                "source": "metadata",
                "asin": row.get("parent_asin"),
                "main_category": row.get("main_category"),
                "categories": row.get("categories"),
            }
        )
    )

# Process review entries
for _, row in review_df.iterrows():
    text_parts = []

    for field in REVIEW_TEXT_FIELDS:
        value = row.get(field)
        if isinstance(value, str) and len(value.strip()) > 0:
            text_parts.append(value.strip())

    if len(text_parts) == 0:
        continue

    full_text = "\n\n".join(text_parts)

    documents.append(
        Document(
            page_content=full_text,
            metadata={
                "source": "review",
                "asin": row.get("asin"),
                "rating": row.get("rating"),
                "verified_purchase": row.get("verified_purchase"),
            }
        )
    )

len(documents)

9999

#### 1.3.1 Selection and justification of fields for retrieval

For retrieval, we focused on fields that contain meaningful natural‑language text.  
* From the metadata, **`title`**, **`description`**, and sometimes **`features`** provide the most useful product information.
* From the reviews, the key fields are **`text`** (full review body) and **`title`** (short summary).  
Other fields like price, images, or ratings don’t add much value for text‑based retrieval. 


#### 1.3.2. Description of text preprocessing

Light preprocessing was applied to keep the text clean but still semantically rich:  
- all text lowercased  
- extra whitespace and URLs removed 
- punctuation kept
- empty or extremely short entries removed

There is no stemming or lemmatization involved, as modern embedding models handle them well on their own.

## 2. Keyword-based retrieval system (BM25)

In [8]:
bm25, tokenized = build_bm25(documents)
save_bm25(bm25, tokenized, save_dir="../bm25_index/")

bm25, tokenized = load_bm25("../bm25_index/")

### 2.1. Example query results

In [9]:
query = "quiet dishwasher stainless steel"
results = bm25_search(query, bm25, documents, k=5)

display_search_results(results, query=query, system_name="BM25 search");

#### BM25 search results for: 'quiet dishwasher stainless steel'

Rank,Score,Source,ASIN,Title,Info,Snippet
1,12.852,metadata,B078T6SMBM,Dishwasher Covers Brushed Stainless Steel Panels,Tools & Home Improvement | Appliances > Parts & Accessories > Dishwasher Parts & Accessories,Dishwasher Covers Brushed Stainless Steel Panels
2,12.443,metadata,B004TSON5M,EASTMAN 8 Stainless Steel Dishwasher Hose 98524,Appliances | Appliances > Parts & Accessories > Dishwasher Parts & Accessories > Hoses,EASTMAN 8 Stainless Steel Dishwasher Hose 98524
3,12.443,metadata,B00NN1CAJW,Front Control Dishwasher in White with Stainless Steel Tub,Tools & Home Improvement | Appliances > Dishwashers > Built-In Dishwashers,Front Control Dishwasher in White with Stainless Steel Tub
4,11.963,review,B013CBE2DE,Relatively easy. The dishwasher is nice and quiet again,"rating=5.0, verified=True",Used a youtube video for how to install. Relatively easy. The dishwasher is nice and quiet again.
5,11.699,metadata,B0C2K57SHK,"EASTMAN 41042 Stainless Steel Dishwasher Connector, 5 Ft, Silver",Tools & Home Improvement | Appliances > Parts & Accessories > Dishwasher Parts & Accessories > Hoses,"EASTMAN 41042 Stainless Steel Dishwasher Connector, 5 Ft, Silver"


## 3. Semantic-based retrival system

### 3.1. Embeddings with sentence transformers

In [10]:
embeddings, model = build_embeddings(documents)
index = build_faiss_index(embeddings)
save_faiss(index, documents, save_dir="../semantic_index/")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

### 3.2. FAISS index and documents

In [11]:
index, documents = load_faiss("../semantic_index/")

### 3.3. Example query results

In [12]:
query = "quiet dishwasher stainless steel"
results = semantic_search(query, index, model, documents, k=5)

display_search_results(results, query=query, system_name="Semantic search");

#### Semantic search results for: 'quiet dishwasher stainless steel'

Rank,Score,Source,ASIN,Title,Info,Snippet
1,0.719,review,B0074CK3UO,poor performer,"rating=2.0, verified=True","as some others have indicated, this dishwasher seems to do very little, if anything, in the way of actually cleaning dishes. :( i don't mind the noise level a bit; i guess i grew up with noisier..."
2,0.722,metadata,B078T6SMBM,Dishwasher Covers Brushed Stainless Steel Panels,Tools & Home Improvement | Appliances > Parts & Accessories > Dishwasher Parts & Accessories,Dishwasher Covers Brushed Stainless Steel Panels
3,0.726,metadata,B00NN1CAJW,Front Control Dishwasher in White with Stainless Steel Tub,Tools & Home Improvement | Appliances > Dishwashers > Built-In Dishwashers,Front Control Dishwasher in White with Stainless Steel Tub
4,0.744,review,B07664NDKB,My dishwasher is finally quiet,"rating=5.0, verified=True","Doing dishes had been a noisy ordeal for a long time. The banging noise was awful. Finally, thanks to Google, I figured out what was wrong and identified this part as the cause and Amazon had it..."
5,0.767,metadata,B01G9ASTG6,"GE CDT835SSJSS Cafe 24"" Stainless Steel Fully Integrated Dishwasher - Energy Star",Appliances | Appliances > Dishwashers > Built-In Dishwashers,"GE CDT835SSJSS Cafe 24"" Stainless Steel Fully Integrated Dishwasher - Energy Star"


## 4. Qualitative evaluation

In [13]:
queries = [
    "stainless steel dishwasher",
    "refrigerator water filter",
    "gas range 30 inch",
    "something to keep drinks cold in a dorm room",
    "appliance for washing dishes quietly at night",
    "replacement part to stop a dishwasher from leaking at the bottom",
    "small machine for drying clothes in an apartment",
    "best dishwasher for a small apartment under $800",
    "energy-efficient refrigerator for a family of four",
    "reliable stove for frequent home cooking that is easy to clean",
]

In [14]:
k = 5

for query in queries:
    bm25_results = bm25_search(query, bm25, documents, k=k)
    semantic_results = semantic_search(query, index, model, documents, k=k)

    bm25_df = display_search_results(
        bm25_results, query=query, system_name="BM25")

    semantic_df = display_search_results(
        semantic_results, query=query, system_name="Semantic")

#### BM25 results for: 'stainless steel dishwasher'

Rank,Score,Source,ASIN,Title,Info,Snippet
1,12.852,metadata,B078T6SMBM,Dishwasher Covers Brushed Stainless Steel Panels,Tools & Home Improvement | Appliances > Parts & Accessories > Dishwasher Parts & Accessories,Dishwasher Covers Brushed Stainless Steel Panels
2,12.443,metadata,B00NN1CAJW,Front Control Dishwasher in White with Stainless Steel Tub,Tools & Home Improvement | Appliances > Dishwashers > Built-In Dishwashers,Front Control Dishwasher in White with Stainless Steel Tub
3,12.443,metadata,B004TSON5M,EASTMAN 8 Stainless Steel Dishwasher Hose 98524,Appliances | Appliances > Parts & Accessories > Dishwasher Parts & Accessories > Hoses,EASTMAN 8 Stainless Steel Dishwasher Hose 98524
4,11.699,metadata,B0052EK0MW,"KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel",Appliances | Appliances > Dishwashers > Built-In Dishwashers,"KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel"
5,11.699,metadata,B0C2K57SHK,"EASTMAN 41042 Stainless Steel Dishwasher Connector, 5 Ft, Silver",Tools & Home Improvement | Appliances > Parts & Accessories > Dishwasher Parts & Accessories > Hoses,"EASTMAN 41042 Stainless Steel Dishwasher Connector, 5 Ft, Silver"


#### Semantic results for: 'stainless steel dishwasher'

Rank,Score,Source,ASIN,Title,Info,Snippet
1,0.536,metadata,B00NN1CAJW,Front Control Dishwasher in White with Stainless Steel Tub,Tools & Home Improvement | Appliances > Dishwashers > Built-In Dishwashers,Front Control Dishwasher in White with Stainless Steel Tub
2,0.563,metadata,B078T6SMBM,Dishwasher Covers Brushed Stainless Steel Panels,Tools & Home Improvement | Appliances > Parts & Accessories > Dishwasher Parts & Accessories,Dishwasher Covers Brushed Stainless Steel Panels
3,0.604,review,B08H4KYL4T,Dishwasher,"rating=5.0, verified=True","Nice to have, Thank You"
4,0.623,metadata,B0052EK0MW,"KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel",Appliances | Appliances > Dishwashers > Built-In Dishwashers,"KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel"
5,0.633,metadata,B01G9ASTG6,"GE CDT835SSJSS Cafe 24"" Stainless Steel Fully Integrated Dishwasher - Energy Star",Appliances | Appliances > Dishwashers > Built-In Dishwashers,"GE CDT835SSJSS Cafe 24"" Stainless Steel Fully Integrated Dishwasher - Energy Star"


#### BM25 results for: 'refrigerator water filter'

...

Rank,Score,Source,ASIN,Title,Info,Snippet
1,0.723,review,B000SDMQOW,Good purchase,"rating=5.0, verified=False",After several months I can say this fridge is awesome. I ordered it sight unseen from a big box store that left me alone to get rid of my old fridge. The local power company hauled away the old...
2,0.743,metadata,B0B6Z36SC1,"KRIB BLING 3.5 Cu.ft Retro Mini Fridge with Freezer - Compact Refrigerator for Home, Office, Dorm, or RV with Adjustable Mechanical Thermostat and 2-Door Design, Wood","Appliances | Appliances > Refrigerators, Freezers & Ice Makers > Refrigerators","KRIB BLING 3.5 Cu.ft Retro Mini Fridge with Freezer - Compact Refrigerator for Home, Office, Dorm, or RV with Adjustable Mechanical Thermostat and 2-Door Design, Wood"
3,0.757,metadata,B093BPW8WY,"FRIGIDAIRE Portable 10L, 15-can Mini Fridge Brushed Stainless Rugged Refrigerator, EFMIS183-SS (Renewed)","Appliances | Appliances > Refrigerators, Freezers & Ice Makers > Refrigerators","FRIGIDAIRE Portable 10L, 15-can Mini Fridge Brushed Stainless Rugged Refrigerator, EFMIS183-SS (Renewed)"
4,0.770,review,B00ZYE4CNI,Inexpensive and have a fourpack,"rating=4.0, verified=True","The delivery was quick and I have installed one of the four. The fridge uses only one filter but I bought the 4 pac pack because it was available. The filter is difficult for me to reach, mounted..."
5,0.801,review,B00O2CF24Q,Bang for the buck!,"rating=5.0, verified=True","Great for the price, keeps fridge from smelling!"


#### BM25 results for: 'reliable stove for frequent home cooking that is easy to clean'

Rank,Score,Source,ASIN,Title,Info,Snippet
1,14.897,metadata,B08YDWSKP6,"Stove Burner Covers, 4 Pack, Gas Stove Burner Liners, Non-stick Reusable Gas Range Stove Protector Covers for Kitchen, Cooking, Cuttable, Easy to Clean JORETLE(Black)",Tools & Home Improvement | Appliances > Parts & Accessories > Range Parts & Accessories > Accessories,"Stove Burner Covers, 4 Pack, Gas Stove Burner Liners, Non-stick Reusable Gas Range Stove Protector Covers for Kitchen, Cooking, Cuttable, Easy to Clean JORETLE(Black)"
2,14.375,review,B07M8JWVN1,Easy to clean stove.,"rating=4.0, verified=True",Easy to clean your stove. The only thing is you need to cut them as per your stove size.
3,12.953,review,B073PLT7TY,Not easy to clean,"rating=5.0, verified=True",For my stove fine. Not easy to clean. Already need new ones.
4,12.543,review,B007EEG3AG,A good fit,"rating=5.0, verified=True","I like the way these stove cover fit (some do not fit well). This helps to keep the clean up after cooking to barely nothing,"
5,12.032,review,B092RXSMGD,Easy to use,"rating=5.0, verified=True",Loved how easy it was to put on and helps to keep the stove clean


#### Semantic results for: 'reliable stove for frequent home cooking that is easy to clean'

Rank,Score,Source,ASIN,Title,Info,Snippet
1,0.691,review,B073PLT7TY,Not easy to clean,"rating=5.0, verified=True",For my stove fine. Not easy to clean. Already need new ones.
2,0.794,review,B0763BMN9T,A must have for your stove,"rating=5.0, verified=True",These make cleaning the stove so easy. Recommend to all.
3,0.811,review,B091TK8WYG,"Great price, work great!","rating=5.0, verified=False",The price on these is great! I have a 4 burner stove so I like having a spare set to swap out with when the other needs cleaning. I throw it in the top rack of the dishwasher and it cleans them...
4,0.814,review,B00UNOSLYU,Great buy,"rating=5.0, verified=True","I really like this cooker, It gets hot very quickly and cooks evenly. It's also very easy to clean up afterwards. Great alternative if you don't have a stovetop available. You just have to buy the..."
5,0.818,review,B0037Z7HQK,Great Cooktop,"rating=5.0, verified=True",Just moved from my beautifully remodeled kitchen with a gas range to an older home with a coiled electric cooktop. This portable Induction burner is now my sole cooktop. I love it. The ease of...
